In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/application_train.csv")

print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows:    307,511
Columns: 122


In [2]:
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
print(df.dtypes.value_counts())
print()
print(df.dtypes)

float64    65
int64      41
object     16
Name: count, dtype: int64

SK_ID_CURR                      int64
TARGET                          int64
NAME_CONTRACT_TYPE             object
CODE_GENDER                    object
FLAG_OWN_CAR                   object
                               ...   
AMT_REQ_CREDIT_BUREAU_DAY     float64
AMT_REQ_CREDIT_BUREAU_WEEK    float64
AMT_REQ_CREDIT_BUREAU_MON     float64
AMT_REQ_CREDIT_BUREAU_QRT     float64
AMT_REQ_CREDIT_BUREAU_YEAR    float64
Length: 122, dtype: object


In [4]:
# Check missing values - how many and what percentage
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_percent': missing_pct
}).query('missing_count > 0').sort_values('missing_percent', ascending=False)

print(f"Total columns:            {df.shape[1]}")
print(f"Columns WITH missing:     {len(missing_df)}")
print(f"Columns WITHOUT missing:  {df.shape[1] - len(missing_df)}")
print()
print(missing_df.to_string())

Total columns:            122
Columns WITH missing:     67
Columns WITHOUT missing:  55

                              missing_count  missing_percent
COMMONAREA_MEDI                      214865             69.9
COMMONAREA_AVG                       214865             69.9
COMMONAREA_MODE                      214865             69.9
NONLIVINGAPARTMENTS_MEDI             213514             69.4
NONLIVINGAPARTMENTS_MODE             213514             69.4
NONLIVINGAPARTMENTS_AVG              213514             69.4
LIVINGAPARTMENTS_MODE                210199             68.4
LIVINGAPARTMENTS_MEDI                210199             68.4
LIVINGAPARTMENTS_AVG                 210199             68.4
FONDKAPREMONT_MODE                   210295             68.4
FLOORSMIN_MODE                       208642             67.8
FLOORSMIN_MEDI                       208642             67.8
FLOORSMIN_AVG                        208642             67.8
YEARS_BUILD_MODE                     204488             6

In [5]:
# Look at our TARGET variable - the class imbalance
target_counts = df['TARGET'].value_counts()
default_rate = df['TARGET'].mean()

print(f"Repaid (0):    {target_counts[0]:,}  ({100 - default_rate*100:.1f}%)")
print(f"Defaulted (1): {target_counts[1]:,}  ({default_rate*100:.1f}%)")
print()
print(f"For every 1 defaulter there are {target_counts[0]//target_counts[1]} repayers")

Repaid (0):    282,686  (91.9%)
Defaulted (1): 24,825  (8.1%)

For every 1 defaulter there are 11 repayers


In [6]:
# Look at the key numeric features
# and compare averages between defaulters vs repayers
key_features = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'DAYS_BIRTH',
    'DAYS_EMPLOYED',
    'EXT_SOURCE_1',
    'EXT_SOURCE_2',
    'EXT_SOURCE_3',
]

print(f"{'Feature':<25} {'Repaid':>10} {'Defaulted':>12} {'Diff %':>8}")
print("-" * 58)

for col in key_features:
    if col in df.columns:
        avg_repaid  = df[df['TARGET']==0][col].mean()
        avg_default = df[df['TARGET']==1][col].mean()
        diff = ((avg_default - avg_repaid) / abs(avg_repaid) * 100)
        print(f"{col:<25} {avg_repaid:>10.2f} {avg_default:>12.2f} {diff:>+7.1f}%")
        

Feature                       Repaid    Defaulted   Diff %
----------------------------------------------------------
AMT_INCOME_TOTAL           169077.72    165611.76    -2.0%
AMT_CREDIT                 602648.28    557778.53    -7.4%
AMT_ANNUITY                 27163.62     26481.74    -2.5%
DAYS_BIRTH                 -16138.18    -14884.83    +7.8%
DAYS_EMPLOYED               65696.15     42394.68   -35.5%
EXT_SOURCE_1                    0.51         0.39   -24.3%
EXT_SOURCE_2                    0.52         0.41   -21.5%
EXT_SOURCE_3                    0.52         0.39   -25.0%


In [7]:
# Check the categorical columns
# and their unique values
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Total categorical columns: {len(categorical_cols)}")
print()

for col in categorical_cols:
    unique_vals = df[col].unique()
    n_unique = len(unique_vals)
    print(f"{col:<30} {n_unique} unique values")
    if n_unique <= 6:
        print(f"   Values: {list(unique_vals)}")

Total categorical columns: 16

NAME_CONTRACT_TYPE             2 unique values
   Values: ['Cash loans', 'Revolving loans']
CODE_GENDER                    3 unique values
   Values: ['M', 'F', 'XNA']
FLAG_OWN_CAR                   2 unique values
   Values: ['N', 'Y']
FLAG_OWN_REALTY                2 unique values
   Values: ['Y', 'N']
NAME_TYPE_SUITE                8 unique values
NAME_INCOME_TYPE               8 unique values
NAME_EDUCATION_TYPE            5 unique values
   Values: ['Secondary / secondary special', 'Higher education', 'Incomplete higher', 'Lower secondary', 'Academic degree']
NAME_FAMILY_STATUS             6 unique values
   Values: ['Single / not married', 'Married', 'Civil marriage', 'Widow', 'Separated', 'Unknown']
NAME_HOUSING_TYPE              6 unique values
   Values: ['House / apartment', 'Rented apartment', 'With parents', 'Municipal apartment', 'Office apartment', 'Co-op apartment']
OCCUPATION_TYPE                19 unique values
WEEKDAY_APPR_PROCESS_START 

In [8]:
# Final EDA summary - everything we've learned
print("=" * 55)
print("EDA COMPLETE — KEY FINDINGS")
print("=" * 55)

# Dataset size
print(f"\nDataset:          {df.shape[0]:,} applicants, {df.shape[1]} columns")

# Target
default_rate = df['TARGET'].mean()
print(f"Default rate:     {default_rate:.1%}  (class imbalance!)")
print(f"Imbalance ratio:  1 defaulter per 11 repayers")

# Missing values
missing = df.isnull().sum()
high_missing = missing[missing/len(df) > 0.5].count()
low_missing  = missing[(missing/len(df) > 0) & (missing/len(df) <= 0.5)].count()
print(f"\nMissing data:")
print(f"  Drop (>50% missing):  {high_missing} columns")
print(f"  Fill (manageable):    {low_missing} columns")
print(f"  Complete:             {df.shape[1] - high_missing - low_missing} columns")

# Data types
print(f"\nFeature types:")
print(f"  Numeric:              {df.select_dtypes(include=[np.number]).shape[1] - 1} columns")
print(f"  Categorical (text):   {df.select_dtypes(include='object').shape[1]} columns")

# Strongest predictors
print(f"\nStrongest predictors found:")
print(f"  EXT_SOURCE_1/2/3     ~25% lower scores for defaulters")
print(f"  DAYS_EMPLOYED        ~35% shorter employment for defaulters")
print(f"  DAYS_BIRTH           younger applicants default more")

# What we'll do next
print(f"\nStep 3 plan:")
print(f"  1. Drop {high_missing} columns with >50% missing")
print(f"  2. Fill remaining missing values")
print(f"  3. Convert age/employment days to years")
print(f"  4. Engineer debt-to-income ratio")
print(f"  5. Encode all 16 categorical columns")
print(f"  6. Save clean dataset ready for modelling")

EDA COMPLETE — KEY FINDINGS

Dataset:          307,511 applicants, 122 columns
Default rate:     8.1%  (class imbalance!)
Imbalance ratio:  1 defaulter per 11 repayers

Missing data:
  Drop (>50% missing):  41 columns
  Fill (manageable):    26 columns
  Complete:             55 columns

Feature types:
  Numeric:              105 columns
  Categorical (text):   16 columns

Strongest predictors found:
  EXT_SOURCE_1/2/3     ~25% lower scores for defaulters
  DAYS_EMPLOYED        ~35% shorter employment for defaulters
  DAYS_BIRTH           younger applicants default more

Step 3 plan:
  1. Drop 41 columns with >50% missing
  2. Fill remaining missing values
  3. Convert age/employment days to years
  4. Engineer debt-to-income ratio
  5. Encode all 16 categorical columns
  6. Save clean dataset ready for modelling
